In [1]:
from langchain_community.document_loaders import TextLoader

## 1. Load Documents

In [14]:
from langchain.document_loaders import PyPDFDirectoryLoader

def load_documents():
    loader = PyPDFDirectoryLoader(r"C:\Users\haris\OneDrive\Desktop\AI\Projects\Chatbot-RAG\data")  #my data Floder
    return loader.load()

docs = load_documents()

print(f"Loaded {len(docs)} documents.")
print(docs[0].page_content[:500])  # preview first 500 characters. not words but characters

Loaded 1133 documents.
Frontiers in Medicine 01 frontiersin.org
Advances in melanoma: 
epidemiology, diagnosis, and 
prognosis
Shayan Waseh 1 and Jason B. Lee 2*
1 Department of Dermatology, Temple University Hospital, Philadelphia, PA, United States, 2 Department 
of Dermatology, Thomas Jefferson University, Philadelphia, PA, United States
Unraveling the multidimensional complexities of melanoma has required 
concerted efforts by dedicated community of researchers and clinicians battling 
against this deadly form of 


## 2. Split the Data into Chunks:

In [16]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,        # adjust based on model context
    chunk_overlap=500       # keeps some context between chunks
)

chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")


print(chunks[23].page_content[:300])  # Print the first 300 characters of the text content from the first chunk of the split document.

Split into 2650 chunks.
Waseh and Lee 10.3389/fmed.2023.1268479
Frontiers in Medicine 08 frontiersin.org
TABLE 3 Melanocytic pathology assessment tool and hierarchy for diagnosis (MPATH-Dx).
Class Risk of tumor 
progression
Probability of 
progression, No. 
per population
Treatment 
recommendation Examples
0 NA NA Consider


## 3. Embedding and Vector DB

In [4]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS



In [17]:
 #1. Load embedding model
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Create vector store
vectorstore = FAISS.from_documents(chunks, embedding)

# 3. Save to disk (optional)
vectorstore.save_local("faiss_index")

### 3.1 See the embeddings

In [6]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector = embedding.embed_documents([chunks[0].page_content])
print(vector[0][:10])  # print first 10 values of the embedding

[-0.05518747493624687, -0.03441521152853966, -0.053741563111543655, -0.033220645040273666, 0.011934067122638226, -0.051758069545030594, 0.024088596925139427, 0.021872056648135185, 0.009093457832932472, -0.09766999632120132]


In [7]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(r"C:\Users\haris\OneDrive\Desktop\AI\Projects\Chatbot-RAG\faiss_index", embedding, allow_dangerous_deserialization=True)  #the folder

# access raw FAISS index vectors
faiss_index = vectorstore.index
import numpy as np

vectors = faiss_index.reconstruct_n(0, faiss_index.ntotal)  # get all vectors
print(np.array(vectors[0])[:10])  # first vector, first 10 dims




# Get all keys from the docstore
keys = list(vectorstore.docstore._dict.keys())
print(keys[0])  # see what the first key is

# Now fetch the actual document
doc = vectorstore.docstore._dict[keys[0]]
print(doc.page_content[:300])

[-0.05518745 -0.03441519 -0.05374157 -0.03322063  0.01193407 -0.05175812
  0.02408861  0.02187206  0.00909345 -0.09767003]
23a377fc-6fe9-4227-8944-c13b01a3d2fe
A hereditary predisposition to breast cancer significantly influences screening and follow-up 
recommendations for high-risk women. However, in patients with a suggestive personal and/or 
family history, a specific predisposing gene is identified in less than 30% of cases. Up to 25% of 
hereditary c


Now we are ready to plug this into a RAG pipeline. We have done:

1. PDF loading and chunking

2. Embedding + FAISS index

3. Verification of both vectors and text

Next step: Run a question-answering chain
We will use LangChain’s RetrievalQA with the loaded FAISS retriever and any LLM (Ollama, OpenAI, etc.)

## 4. Retrieval

In [8]:
#from langchain_community.chat_models import ChatOllama
#from langchain.chains import RetrievalQA

#llm = ChatOllama(model="mistral")  # uses localhost:11434 by default

#qa = RetrievalQA.from_chain_type(
   # llm=llm,
   # retriever=vectorstore.as_retriever(search_kwargs={"k": 8})
##)

#response = qa.run("Who is Napoleon in this?")
##print(response)

In [10]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatOllama

# Define your custom prompt template
template = """
You are a biomedical assistant for melanoma. Using the information below, answer the question in a clear, informative, and helpful manner.
If the answer is not directly present, use reasoning based on context.

Context:
{context}

Question:
{question}

Helpful Answer:
"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

# Load LLM
llm = ChatOllama(model="mistral")

# Create the chain
qa_chain = LLMChain(llm=llm, prompt=prompt)

# Do retrieval manually
retrieved_docs = vectorstore.similarity_search("What is melanoma?", k=5)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Run the chain
response = qa_chain.run({"context": context, "question": "What is melanoma?"})
print(response)


 Melanoma is a type of aggressive cancer that develops in pigment-producing cells called melanocytes. It's the most dangerous form of skin cancer, although it accounts for only 2% of all skin cancer types. The incidence of melanoma has been increasing worldwide and can arise in different anatomical sites, such as the skin (cutaneous melanoma), eye (uveal melanoma), mucosal membranes, or even the central nervous system. Melanoma is often caused by exposure to ultraviolet radiation from the sun and can lead to a variety of genetic alterations that promote unregulated cell proliferation. Early stage melanoma can be cured with surgical excision, while late stage melanoma requires various treatments including surgery, radiation, immunotherapy, targeted therapy, chemotherapy, and innovative approaches like nanotechnology.


## 5. Trying different Queries to see whether it holds context:

In [ ]:
response = qa.run("are there any new and innovative treatment options for melanoma?")
print(response)

NameError: name 'qa' is not defined

In [ ]:
from langchain.chains import RetrievalQAWithSourcesChain

qa = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4})
)

response = qa({"question": "What is melanoma and how is it treated?"})
print("Answer:", response["answer"])
print("Sources:", response["sources"])

Answer:  Melanoma is a type of skin cancer that can also occur in other parts of the body such as the eye, mucosal membranes, or central nervous system. It's the most aggressive form of skin cancer and its incidence is increasing globally. The primary cause is UV radiation and sunburns, although additional risk factors include aging (due to DNA repair decline) and genetic predisposition (germline mutations in CDKN2A, CDK4, MITF, TERT, MC1R). There are four stages of melanoma based on the TNM system, with stage I-II being localized, stage III having node-positive disease, and stage IV indicating metastatic disease. Treatment for early-stage melanoma usually involves surgery, which is often curative. Advanced melanoma can be difficult to treat, with chemotherapy, radiotherapy, and older targeted therapies being less effective. However, recent advances in immunotherapy (PD-1, combination therapy) and targeted therapy (BRAF/MEK inhibitors) have significantly improved survival rates, partic

k = 4 means the retriever fetches the top 4 most relevant chunks (based on embedding similarity) from the vector store, not four separate answers. These 4 chunks are combined into a single prompt and passed to the LLM, giving it broader context to generate a single, more accurate answer.

In [ ]:
response = qa("what is gene therapy for melanoma? is it expensive?")
print(response)

{'question': 'what is gene therapy for melanoma? is it expensive?', 'answer': " Gene therapy for melanoma is a developing treatment modality that aims to improve prognosis and cut down on the cost of treatment. The specific cost of gene therapy for melanoma is not mentioned in the provided sources, but it's stated that personalized treatment strategies will become more effective as technologies like genetic sequencing and bioinformatics continue to evolve (", 'sources': "C:\\Users\\haris\\OneDrive\\Desktop\\AI\\Projects\\Chatbot-RAG\\data\\cancers-17-00707.pdf). It's also mentioned that advances in understanding the molecular characteristics of melanoma have led to the development of more effective therapies, giving hope that a cure may be within reach ("}
